# Lesson 3: pandas

**Week 3 · Data Engineering Course**

---

pandas is the most widely used Python library for working with tabular data. It lets you load a CSV into a **DataFrame** — a table with named columns — and then filter, clean, and summarise it with just a few lines of code.

**Install before starting** (run once in your terminal):
```
pip install pandas
```

**What you will learn:**
- Loading a CSV into a DataFrame
- Exploring: shape, types, missing values, unique values, statistics
- Selecting columns and filtering rows
- Cleaning: whitespace, casing, typos, currency symbols, missing values, duplicates
- Grouping and summarising
- Writing the clean result to a new CSV

In [ ]:
import pandas as pd
from pathlib import Path

RAW   = Path('../data/raw')
CLEAN = Path('../data/clean')
CLEAN.mkdir(parents=True, exist_ok=True)

print(f'pandas version: {pd.__version__}')

---

## 3.1 Loading a CSV

`pd.read_csv()` reads a CSV file into a DataFrame.

In [ ]:
df = pd.read_csv(RAW / 'sales_q1.csv')

print(f'Shape: {df.shape}')         # (rows, columns)
print(f'Columns: {list(df.columns)}')

---

## 3.2 Exploring the Data

Before cleaning anything, look at the data. These steps take under a minute and tell you almost everything you need.

In [ ]:
df.head()    # first 5 rows

In [ ]:
df.tail(3)   # last 3 rows — useful for checking that the file did not get cut off

In [ ]:
df.info()    # column names, non-null counts, data types

In [ ]:
# df.describe() gives statistics for every numeric column
# count, mean, min, max, and percentiles
df.describe()

In [ ]:
# Missing values per column
print(df.isnull().sum())

In [ ]:
# Unique values in the category column — quickly reveals casing problems and typos
print(df['category'].value_counts())
print(f'\nUnique categories: {df["category"].nunique()}')

In [ ]:
# df.sample() shows random rows — less biased than head() which always shows the top
# Useful for spotting problems that only appear partway through the file
df.sample(5, random_state=42)   # random_state=42 makes it repeatable

---

## 3.3 Selecting Columns and Filtering Rows

In [ ]:
# Select one column — returns a Series
print(df['category'].head())

# Select multiple columns — returns a DataFrame
subset = df[['order_id', 'customer_name', 'product', 'total']]
subset.head()

In [ ]:
# Filter rows: keep only Electronics
mask = df['category'].str.strip().str.upper() == 'ELECTRONICS'
electronics = df[mask]
print(f'Electronics rows: {len(electronics)}')
electronics[['order_id', 'product', 'category']].head()

In [ ]:
# .isin() filters by a list of values — cleaner than writing multiple == conditions
wanted_categories = ['Electronics', 'Clothing']
mask = df['category'].str.strip().str.title().isin(wanted_categories)
print(f'Rows in Electronics or Clothing: {mask.sum()}')
df[mask][['order_id', 'product', 'category']].head(8)

In [ ]:
# Combine conditions with & (AND) — each condition MUST be in parentheses
high_value = df[
    (df['category'].str.strip().str.upper() == 'ELECTRONICS') &
    (pd.to_numeric(df['total'], errors='coerce') > 200)
]
print(f'High-value Electronics rows: {len(high_value)}')
high_value[['order_id', 'product', 'total']].head()

In [ ]:
# .sort_values() sorts the result
# Useful after a filter to see the best or worst rows
top_orders = (
    df[['order_id', 'product', 'total']]
    .assign(total=pd.to_numeric(df['total'], errors='coerce'))
    .sort_values('total', ascending=False)
    .head(5)
)
print('Top 5 orders by total:')
print(top_orders)

---

## 3.4 Cleaning Data

Each step fixes one type of problem. Always work on a copy.

In [ ]:
clean = df.copy()

# Step 1: Remove duplicates
before = len(clean)
clean = clean.drop_duplicates(subset=['order_id'])
print(f'Duplicates removed: {before - len(clean)}')

In [ ]:
# Step 2: Strip whitespace from text columns
clean['customer_name'] = clean['customer_name'].str.strip()
clean['product']       = clean['product'].str.strip()
print('Whitespace stripped.')

In [ ]:
# Step 3: Standardise category — strip, title case
print('Before:', clean['category'].value_counts().to_dict())
clean['category'] = clean['category'].str.strip().str.title()
print('After:', clean['category'].value_counts().to_dict())

In [ ]:
# Step 4: Fix category typos using a dict passed to .replace()
# .replace() with a dict maps old values -> new values across the whole column
TYPO_MAP = {
    'Electrnics':    'Electronics',
    'Home & Kithen': 'Home & Kitchen',
}
clean['category'] = clean['category'].replace(TYPO_MAP)
print('After typo fix:', clean['category'].value_counts().to_dict())

In [ ]:
# Step 5: Clean unit_price — remove $, convert to float
clean['unit_price'] = (
    clean['unit_price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
)
clean['unit_price'] = pd.to_numeric(clean['unit_price'], errors='coerce')
print(f'unit_price dtype: {clean["unit_price"].dtype}')
print(f'Null unit_prices: {clean["unit_price"].isnull().sum()}')

In [ ]:
# Step 6: Convert quantity — Q3 data stores it as float (1.0, 2.0)
# Convert to numeric first, then to a nullable integer
clean['quantity'] = pd.to_numeric(clean['quantity'], errors='coerce')
print(f'quantity dtype before: {clean["quantity"].dtype}')
clean['quantity'] = clean['quantity'].astype('Int64')   # capital I = nullable integer
print(f'quantity dtype after:  {clean["quantity"].dtype}')
print(clean['quantity'].head(6))

In [ ]:
# Step 7: Fill missing totals
clean['total'] = pd.to_numeric(clean['total'], errors='coerce')
missing_before = clean['total'].isnull().sum()
clean['total'] = clean['total'].fillna(clean['quantity'] * clean['unit_price'])
print(f'Missing totals: {missing_before} before, {clean["total"].isnull().sum()} after')

In [ ]:
# Step 8: Drop invalid rows
before = len(clean)
clean = clean[clean['quantity'] > 0]
print(f'Rows dropped (quantity <= 0): {before - len(clean)}')
print(f'Clean rows: {len(clean)}')

---

## 3.5 Grouping and Summarising

Groupby works like SQL `GROUP BY`: split the data by a column, aggregate each group, combine the results.

In [ ]:
# Total revenue and order count by category
revenue = (
    clean
    .groupby('category')['total']
    .agg(orders='count', revenue='sum')
    .sort_values('revenue', ascending=False)
    .round(2)
)
print(revenue)

In [ ]:
# Multiple aggregation functions: count, sum, mean, min, max
richer = (
    clean
    .groupby('category')['total']
    .agg(
        orders   = 'count',
        revenue  = 'sum',
        avg_order = 'mean',
        min_order = 'min',
        max_order = 'max'
    )
    .round(2)
    .sort_values('revenue', ascending=False)
)
print(richer)

In [ ]:
# Group by two columns: category AND sales_rep
by_cat_rep = (
    clean
    .groupby(['category', 'sales_rep'])
    .agg(
        orders  = ('order_id', 'count'),
        revenue = ('total', 'sum')
    )
    .round(2)
    .sort_values('revenue', ascending=False)
)
print(by_cat_rep.head(8))

In [ ]:
# Sort the groupby result a different way
# Most orders first
by_orders = (
    clean
    .groupby('sales_rep')['order_id']
    .count()
    .rename('orders')
    .sort_values(ascending=False)
)
print('Sales reps by number of orders:')
print(by_orders)

---

## 3.6 Writing to CSV

In [ ]:
# Always use index=False so pandas does not write row numbers as a column
out_path = CLEAN / 'sales_q1_clean.csv'
clean.to_csv(out_path, index=False)
print(f'Saved {len(clean)} rows to {out_path}')

# Verify by reading back
check = pd.read_csv(out_path)
print(f'Re-read: {check.shape}')
check.head()

---

## Key Takeaways

1. `pd.read_csv()` loads a CSV into a DataFrame. Always check `.shape`, `.info()`, `.describe()`, `.isnull().sum()`, and `.value_counts()` before cleaning.
2. `df.sample(5)` shows random rows — less biased than `.head()` for spotting scattered problems.
3. Filter rows with a boolean mask. Use `.isin([list])` when matching against several values. Wrap every condition in parentheses when combining with `&` or `|`.
4. `.sort_values('column')` sorts results. Use `ascending=False` for largest first.
5. Always call `.copy()` before modifying a DataFrame so the original is preserved.
6. `.replace({old: new})` corrects typos across a whole column in one line.
7. `pd.to_numeric(series, errors='coerce')` converts to numbers and turns unparseable values into NaN. Use `astype('Int64')` (capital I) for a nullable integer column.
8. `.groupby().agg()` is the pandas equivalent of SQL `GROUP BY`. Pass a dict to `agg()` to compute count, sum, mean, min, and max all at once.
9. Always save with `index=False`.